In [1]:
import pandas as pd
import os
import sys

sys.path.append(os.path.join(os.getcwd(), ".."))

from system_ranking import rank_systems
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

# WMT Significance by LP

In [2]:
model2info = {
    "gpt-4o-no-context": {
        "alias": "gpt-4o-no-context",
    },
    "gpt-4o-context": {
        "alias": "gpt-4o-context",
    },
    "greedy-7b-wo-context": {
        "alias": "TowerInstruct-7B w/o context",
    },
    "greedy-7b-w-context": {
        "alias": "TowerInstruct-7B w/ context",
    },
    "mbr-instruct": {
        "alias": "TowerInstruct-7B w/o context + COMET MBR",
    },
    "mbr-source-instruct": {
        "alias": "TowerInstruct-7B w/o context + ContextCOMET MBR",
    },
    "greedy-7b-ft-wo-context": {
        "alias": "TowerChat-7B w/o context",
    },
    "greedy-7b-ft-w-context": {
        "alias": "TowerChat-7B w/ context",
    },
    "greedy-7b-ft-w-context-english": {
        "alias": "TowerChat-7B w/ context (English)",
    },
    "mbr": {
        "alias": "TowerChat-7B w/ context + COMET MBR",
    },
    "mbr-source": {
        "alias": "TowerChat-7B w/ context + ContextCOMET MBR",
    },
    "towerchat-ft-7b-wo-context": {
        "alias": "TowerChat-7B SFT on QAD w/o context",
    },
    "towerchat-ft-7b-w-context": {
        "alias": "TowerChat-7B SFT on QAD w/ context",
    },
}

models = list(model2info.keys())

In [3]:
def load_results(results_df, model, lp, metric):
    metric_score = results_df[results_df["lp"] == lp].loc[:, f"{model}-{metric}"].mean()
    return {
        "model": model,
        "lp": lp,
        metric: metric_score,
    }


def load_segment_results(results_df, model, lp, metric):
    return results_df[results_df["lp"] == lp].loc[:, f"{model}-{metric}"].tolist()

In [4]:
metric = "comet"
lps = [
    "en-de",
    "en-fr",
    "en-pt-br",
    "en-ko",
    "en-nl",
    "de-en",
    "fr-en",
    "pt-br-en",
    "ko-en",
    "nl-en",
]
lp_names = [
    "en-de",
    "en-fr",
    "en-pt",
    "en-ko",
    "en-nl",
    "en-de",
    "en-fr",
    "en-pt",
    "en-ko",
    "en-nl",
]
results = pd.concat(
    [
        pd.DataFrame(
            [
                load_results(
                    pd.read_csv(f"../tacl_review/wmt24_chat/test.{lp_name}.csv"),
                    model,
                    lp,
                    metric,
                )
                for lp, lp_name in zip(lps, lp_names)
            ]
        )
        for model in models
    ]
)
pivot_results = results.pivot(columns="lp", index="model", values=metric)
pivot_results = pivot_results.loc[models]

clusters_by_lp = {}
threshold_p_value = 0.05
for lp, lp_name in zip(lps, lp_names):
    data = {
        m: load_segment_results(
            pd.read_csv(f"../tacl_review/wmt24_chat/test.{lp_name}.csv"), m, lp, metric
        )
        for m in models
    }
    seg_scores = pd.DataFrame(data)  # for metricx, lower is better

    clusters = rank_systems(seg_scores, threshold_p_value=threshold_p_value)

    clusters_by_lp[lp] = clusters

clusters = pd.DataFrame(clusters_by_lp)
clusters

,en-de,en-fr,en-pt-br,en-ko,en-nl,de-en,fr-en,pt-br-en,ko-en,nl-en
gpt-4o-no-context,3.0,4.0,7.0,7.0,5.0,4.0,5.0,4.0,6.0,3.0
gpt-4o-context,4.0,3.0,6.0,5.0,4.0,5.0,5.0,5.0,8.0,5.0
greedy-7b-wo-context,6.0,5.0,8.0,8.0,7.0,5.0,4.0,7.0,6.0,5.0
greedy-7b-w-context,6.0,6.0,9.0,9.0,7.0,6.0,5.0,6.0,7.0,6.0
mbr-instruct,2.0,3.0,6.0,5.0,5.0,1.0,1.0,5.0,5.0,3.0
mbr-source-instruct,4.0,4.0,7.0,6.0,6.0,2.0,2.0,6.0,5.0,4.0
greedy-7b-ft-wo-context,5.0,4.0,5.0,4.0,5.0,4.0,4.0,5.0,4.0,4.0
greedy-7b-ft-w-context,3.0,3.0,3.0,3.0,3.0,4.0,4.0,3.0,3.0,3.0
greedy-7b-ft-w-context-english,2.0,3.0,4.0,3.0,5.0,4.0,4.0,4.0,3.0,3.0
mbr,1.0,1.0,1.0,1.0,1.0,2.0,3.0,1.0,1.0,1.0


In [5]:
cluster_dict = {
    "1.0": "firstcluster",
    "2.0": "secondcluster",
    "3.0": "thirdcluster",
    "4.0": "fourthcluster",
    "5.0": "fifthcluster",
    "6.0": "sixthcluster",
    "7.0": "seventhcluster",
    "8.0": "eighthcluster",
    "9.0": "ninthcluster",
    "10.0": "tenthcluster",
}
# join with latex
latex_results = (pivot_results * 100).round(2)
# sort latex_results columns to match lps
latex_results = latex_results[lps]
for model in models:
    print(model, end=": ")
    for lp in lps:
        if lp == "en-nl":
            end = " && "
        elif lp == "nl-en":
            end = " \\\\"
        else:
            end = " & "
        print(
            f"{latex_results.loc[model, lp]}\\{cluster_dict[str(clusters.loc[model, lp])]}",
            end=end,
        )
    print("\n")

gpt-4o-no-context: 92.74\thirdcluster & 92.43\fourthcluster & 93.01\seventhcluster & 92.26\seventhcluster & 92.68\fifthcluster && 92.16\fourthcluster & 92.18\fifthcluster & 91.4\fourthcluster & 93.24\sixthcluster & 93.07\thirdcluster \\

gpt-4o-context: 92.49\fourthcluster & 92.62\thirdcluster & 93.4\sixthcluster & 93.06\fifthcluster & 93.08\fourthcluster && 91.95\fifthcluster & 91.76\fifthcluster & 91.19\fifthcluster & 90.67\eighthcluster & 92.39\fifthcluster \\

greedy-7b-wo-context: 91.71\sixthcluster & 91.89\fifthcluster & 91.9\eighthcluster & 91.64\eighthcluster & 91.3\seventhcluster && 92.08\fifthcluster & 92.78\fourthcluster & 90.43\seventhcluster & 93.13\sixthcluster & 92.45\fifthcluster \\

greedy-7b-w-context: 91.48\sixthcluster & 91.08\sixthcluster & 90.79\ninthcluster & 91.13\ninthcluster & 91.0\seventhcluster && 91.33\sixthcluster & 91.89\fifthcluster & 90.63\sixthcluster & 91.88\seventhcluster & 91.08\sixthcluster \\

mbr-instruct: 92.77\secondcluster & 92.62\thirdcluster

# Aggregate significance

In [6]:
metrics = ["comet", "metricx"]  # "context-comet-qe"
lps = [
    "en-de",
    "en-fr",
    "en-pt-br",
    "en-ko",
    "en-nl",
    "de-en",
    "fr-en",
    "pt-br-en",
    "ko-en",
    "nl-en",
]
lp_names = [
    "en-de",
    "en-fr",
    "en-pt",
    "en-ko",
    "en-nl",
    "en-de",
    "en-fr",
    "en-pt",
    "en-ko",
    "en-nl",
]
from_en_lps = [lp for lp in lps if lp.startswith("en-")]
into_en_lps = [lp for lp in lps if lp.endswith("-en")]
for metric in metrics:
    print(metric)
    results = pd.concat(
        [
            pd.DataFrame(
                [
                    load_results(
                        pd.read_csv(f"../tacl_review/wmt24_chat/test.{lp_name}.csv"),
                        model,
                        lp,
                        metric,
                    )
                    for lp, lp_name in zip(lps, lp_names)
                ]
            )
            for model in models
        ]
    )
    if metric == "metricx":
        results[metric] = -results[metric]
    clusters_by_lp = {}
    threshold_p_value = 0.05
    for lp, lp_name in zip(lps, lp_names):
        data = {
            m: load_segment_results(
                pd.read_csv(f"../tacl_review/wmt24_chat/test.{lp_name}.csv"),
                m,
                lp,
                metric,
            )
            for m in models
        }
        seg_scores = pd.DataFrame(data)
        if metric == "metricx":
            seg_scores = -seg_scores

        clusters = rank_systems(seg_scores, threshold_p_value=threshold_p_value)

        clusters_by_lp[lp] = clusters

    clusters = pd.DataFrame(clusters_by_lp)

    # aggregate and get bordas
    pivot_results = results.pivot(columns="lp", index="model", values=metric)
    pivot_results = pivot_results.loc[models]

    into_en_results = pivot_results.loc[:, into_en_lps].mean(axis=1)
    from_en_results = pivot_results.loc[:, from_en_lps].mean(axis=1)

    from_en_borda_count = (
        (clusters[from_en_lps].sum(axis=1).sort_values() / len(from_en_lps))
        .round(0)
        .loc[models]
    )
    into_en_borda_count = (
        (clusters[into_en_lps].sum(axis=1).sort_values() / len(into_en_lps))
        .round(0)
        .loc[models]
    )
    print(f"From EN borda counts")
    display(from_en_borda_count)
    print(f"Into EN borda counts")
    display(into_en_borda_count)

comet
From EN borda counts


gpt-4o-no-context                 5.0
gpt-4o-context                    4.0
greedy-7b-wo-context              7.0
greedy-7b-w-context               7.0
mbr-instruct                      4.0
mbr-source-instruct               5.0
greedy-7b-ft-wo-context           5.0
greedy-7b-ft-w-context            3.0
greedy-7b-ft-w-context-english    3.0
mbr                               1.0
mbr-source                        2.0
towerchat-ft-7b-wo-context        4.0
towerchat-ft-7b-w-context         3.0
dtype: float64

Into EN borda counts


gpt-4o-no-context                 4.0
gpt-4o-context                    6.0
greedy-7b-wo-context              5.0
greedy-7b-w-context               6.0
mbr-instruct                      3.0
mbr-source-instruct               4.0
greedy-7b-ft-wo-context           4.0
greedy-7b-ft-w-context            3.0
greedy-7b-ft-w-context-english    4.0
mbr                               2.0
mbr-source                        3.0
towerchat-ft-7b-wo-context        4.0
towerchat-ft-7b-w-context         3.0
dtype: float64

metricx
From EN borda counts


gpt-4o-no-context                 5.0
gpt-4o-context                    3.0
greedy-7b-wo-context              4.0
greedy-7b-w-context               6.0
mbr-instruct                      3.0
mbr-source-instruct               3.0
greedy-7b-ft-wo-context           3.0
greedy-7b-ft-w-context            3.0
greedy-7b-ft-w-context-english    3.0
mbr                               2.0
mbr-source                        2.0
towerchat-ft-7b-wo-context        3.0
towerchat-ft-7b-w-context         3.0
dtype: float64

Into EN borda counts


gpt-4o-no-context                 3.0
gpt-4o-context                    4.0
greedy-7b-wo-context              4.0
greedy-7b-w-context               4.0
mbr-instruct                      3.0
mbr-source-instruct               3.0
greedy-7b-ft-wo-context           3.0
greedy-7b-ft-w-context            2.0
greedy-7b-ft-w-context-english    2.0
mbr                               1.0
mbr-source                        1.0
towerchat-ft-7b-wo-context        2.0
towerchat-ft-7b-w-context         2.0
dtype: float64

### chrf particular case

In [7]:
from sacrebleu import corpus_chrf
from tqdm import tqdm

n_iters = 100
pct_dataset = 0.5


def load_results_chrf(results_df, model, lp, metric):
    lp_subset = results_df[results_df["lp"] == lp]
    references = [str(s) for s in lp_subset["reference"].tolist()]
    hypotheses = [str(s) for s in lp_subset[f"{model}"].tolist()]
    metric_score = corpus_chrf(hypotheses, [references]).score
    return {
        "model": model,
        "lp": lp,
        metric: metric_score,
    }


def load_segment_results_chrf(results_df, model, lp, n_iters, pct_dataset):
    lp_subset = results_df[results_df["lp"] == lp]
    scores = []
    for i in tqdm(range(n_iters), total=n_iters, desc=f"Bootstrapping chrf for {lp}"):
        # make sure we test on different subsets for different models
        random_subset = lp_subset.sample(frac=pct_dataset, random_state=i).reset_index(
            drop=True
        )
        references = [str(s) for s in random_subset["reference"].tolist()]
        hypotheses = [str(s) for s in random_subset[f"{model}"].tolist()]
        scores.append(corpus_chrf(hypotheses, [references]).score)

    return scores


lps = [
    "en-de",
    "en-fr",
    "en-pt-br",
    "en-ko",
    "en-nl",
    "de-en",
    "fr-en",
    "pt-br-en",
    "ko-en",
    "nl-en",
]
lp_names = [
    "en-de",
    "en-fr",
    "en-pt",
    "en-ko",
    "en-nl",
    "en-de",
    "en-fr",
    "en-pt",
    "en-ko",
    "en-nl",
]
from_en_lps = [lp for lp in lps if lp.startswith("en-")]
into_en_lps = [lp for lp in lps if lp.endswith("-en")]
metric = "chrf"
results = pd.concat(
    [
        pd.DataFrame(
            [
                load_results_chrf(
                    pd.read_csv(f"../tacl_review/wmt24_chat/test.{lp_name}.csv"),
                    model,
                    lp,
                    metric,
                )
                for lp, lp_name in zip(lps, lp_names)
            ]
        )
        for model in models
    ]
)
if metric == "metricx":
    results[metric] = -results[metric]
clusters_by_lp = {}
threshold_p_value = 0.05
for lp, lp_name in zip(lps, lp_names):
    data = {
        m: load_segment_results_chrf(
            pd.read_csv(f"../tacl_review/wmt24_chat/test.{lp_name}.csv"),
            m,
            lp,
            n_iters,
            pct_dataset,
        )
        for m in models
    }
    seg_scores = pd.DataFrame(data)
    if metric == "metricx":
        seg_scores = -seg_scores

    clusters = rank_systems(seg_scores, threshold_p_value=threshold_p_value)

    clusters_by_lp[lp] = clusters

clusters = pd.DataFrame(clusters_by_lp)

# aggregate and get bordas
pivot_results = results.pivot(columns="lp", index="model", values=metric)
pivot_results = pivot_results.loc[models]
into_en_results = pivot_results.loc[:, into_en_lps].mean(axis=1)
from_en_results = pivot_results.loc[:, from_en_lps].mean(axis=1)

from_en_borda_count = (
    (clusters[from_en_lps].sum(axis=1).sort_values() / len(from_en_lps))
    .round(0)
    .loc[models]
)
into_en_borda_count = (
    (clusters[into_en_lps].sum(axis=1).sort_values() / len(into_en_lps))
    .round(0)
    .loc[models]
)
print(f"From EN borda counts")
display(from_en_borda_count)
print(f"Into EN borda counts")
display(into_en_borda_count)

Bootstrapping chrf for nl-en: 100%|██████████| 100/100 [00:05<00:00, 19.05it/s]


From EN borda counts


gpt-4o-no-context                  7.0
gpt-4o-context                     7.0
greedy-7b-wo-context              10.0
greedy-7b-w-context               12.0
mbr-instruct                      10.0
mbr-source-instruct               10.0
greedy-7b-ft-wo-context            7.0
greedy-7b-ft-w-context             4.0
greedy-7b-ft-w-context-english     4.0
mbr                                2.0
mbr-source                         2.0
towerchat-ft-7b-wo-context         6.0
towerchat-ft-7b-w-context          3.0
dtype: float64

Into EN borda counts


gpt-4o-no-context                  5.0
gpt-4o-context                     8.0
greedy-7b-wo-context               7.0
greedy-7b-w-context               10.0
mbr-instruct                       8.0
mbr-source-instruct                7.0
greedy-7b-ft-wo-context            5.0
greedy-7b-ft-w-context             3.0
greedy-7b-ft-w-context-english     3.0
mbr                                2.0
mbr-source                         2.0
towerchat-ft-7b-wo-context         4.0
towerchat-ft-7b-w-context          3.0
dtype: float64

# BCONTRAST

In [14]:
model2info = {
    "gpt-4o-no-context": {
        "alias": "gpt-4o-no-context",
    },
    "gpt-4o-context": {
        "alias": "gpt-4o-context",
    },
    "greedy-7b-wo-context": {
        "alias": "TowerInstruct-7B w/o context",
    },
    "greedy-7b-w-context": {
        "alias": "TowerInstruct-7B w/ context",
    },
    "mbr-instruct": {
        "alias": "TowerInstruct-7B w/o context + COMET MBR",
    },
    "mbr-source-instruct": {
        "alias": "TowerInstruct-7B w/o context + ContextCOMET MBR",
    },
    "greedy-7b-ft-wo-context": {
        "alias": "TowerChat-7B w/o context",
    },
    "greedy-7b-ft-w-context": {
        "alias": "TowerChat-7B w/ context",
    },
    "greedy-7b-ft-w-context-english": {
        "alias": "TowerChat-7B w/ context (English)",
    },
    "mbr-chat": {
        "alias": "TowerChat-7B w/ context + COMET MBR",
    },
    "mbr-source-chat": {
        "alias": "TowerChat-7B w/ context + ContextCOMET MBR",
    },
    "towerchat-ft-7b-wo-context": {
        "alias": "TowerChat-7B SFT on QAD w/o context",
    },
    "towerchat-ft-7b-w-context": {
        "alias": "TowerChat-7B SFT on QAD w/ context",
    },
}

models = list(model2info.keys())

In [15]:
info_by_metric = {
    "chrf": {"latex_results": 0, "clusters": 0},
    "comet": {"latex_results": 0, "clusters": 0},
    "metricx": {"latex_results": 0, "clusters": 0},
}

In [17]:
metrics = ["comet", "metricx"]
for metric in metrics:
    print(metric)
    lps = [
        "en-de",
        "de-en",
    ]
    lp_names = [
        "en-de",
        "en-de",
    ]
    results = pd.concat(
        [
            pd.DataFrame(
                [
                    load_results(
                        pd.read_csv(f"../tacl_review/bcontrast/test.{lp_name}.csv"),
                        model,
                        lp,
                        metric,
                    )
                    for lp, lp_name in zip(lps, lp_names)
                ]
            )
            for model in models
        ]
    )
    if metric == "metricx":
        results[metric] = -results[metric]
    pivot_results = results.pivot(columns="lp", index="model", values=metric)
    pivot_results = pivot_results.loc[models]

    clusters_by_lp = {}
    threshold_p_value = 0.05
    for lp, lp_name in zip(lps, lp_names):
        data = {
            m: load_segment_results(
                pd.read_csv(f"../tacl_review/bcontrast/test.{lp_name}.csv"),
                m,
                lp,
                metric,
            )
            for m in models
        }
        seg_scores = pd.DataFrame(data)  # for metricx, lower is better
        if metric == "metricx":
            seg_scores = -seg_scores

        clusters = rank_systems(seg_scores, threshold_p_value=threshold_p_value)

        clusters_by_lp[lp] = clusters

    clusters = pd.DataFrame(clusters_by_lp)
    clusters

    cluster_dict = {
        "1.0": "firstcluster",
        "2.0": "secondcluster",
        "3.0": "thirdcluster",
        "4.0": "fourthcluster",
        "5.0": "fifthcluster",
        "6.0": "sixthcluster",
        "7.0": "seventhcluster",
        "8.0": "eighthcluster",
        "9.0": "ninthcluster",
        "10.0": "tenthcluster",
    }
    # join with latex
    if metric == "comet":
        latex_results = (pivot_results * 100).round(2)
    elif metric == "metricx":
        latex_results = -pivot_results.round(2)
    # sort latex_results columns to match lps
    latex_results = latex_results[lps]
    info_by_metric[metric] = {
        "latex_results": latex_results,
        "clusters": clusters,
    }

    into_en_results = pivot_results.loc[:, into_en_lps].mean(axis=1)
    from_en_results = pivot_results.loc[:, from_en_lps].mean(axis=1)

    from_en_borda_count = (
        (clusters[from_en_lps].sum(axis=1).sort_values() / len(from_en_lps))
        .round(0)
        .loc[models]
    )
    into_en_borda_count = (
        (clusters[into_en_lps].sum(axis=1).sort_values() / len(into_en_lps))
        .round(0)
        .loc[models]
    )
    print(f"From EN borda counts")
    display(from_en_borda_count)
    print(f"Into EN borda counts")
    display(into_en_borda_count)

comet
From EN borda counts


gpt-4o-no-context                 2.0
gpt-4o-context                    1.0
greedy-7b-wo-context              6.0
greedy-7b-w-context               6.0
mbr-instruct                      3.0
mbr-source-instruct               4.0
greedy-7b-ft-wo-context           5.0
greedy-7b-ft-w-context            4.0
greedy-7b-ft-w-context-english    4.0
mbr-chat                          1.0
mbr-source-chat                   2.0
towerchat-ft-7b-wo-context        4.0
towerchat-ft-7b-w-context         3.0
dtype: float64

Into EN borda counts


gpt-4o-no-context                 2.0
gpt-4o-context                    1.0
greedy-7b-wo-context              4.0
greedy-7b-w-context               4.0
mbr-instruct                      2.0
mbr-source-instruct               3.0
greedy-7b-ft-wo-context           5.0
greedy-7b-ft-w-context            4.0
greedy-7b-ft-w-context-english    4.0
mbr-chat                          2.0
mbr-source-chat                   3.0
towerchat-ft-7b-wo-context        5.0
towerchat-ft-7b-w-context         5.0
dtype: float64

metricx
From EN borda counts


gpt-4o-no-context                 4.0
gpt-4o-context                    1.0
greedy-7b-wo-context              4.0
greedy-7b-w-context               5.0
mbr-instruct                      2.0
mbr-source-instruct               2.0
greedy-7b-ft-wo-context           3.0
greedy-7b-ft-w-context            2.0
greedy-7b-ft-w-context-english    3.0
mbr-chat                          1.0
mbr-source-chat                   1.0
towerchat-ft-7b-wo-context        2.0
towerchat-ft-7b-w-context         2.0
dtype: float64

Into EN borda counts


gpt-4o-no-context                 1.0
gpt-4o-context                    1.0
greedy-7b-wo-context              4.0
greedy-7b-w-context               3.0
mbr-instruct                      2.0
mbr-source-instruct               2.0
greedy-7b-ft-wo-context           4.0
greedy-7b-ft-w-context            3.0
greedy-7b-ft-w-context-english    2.0
mbr-chat                          2.0
mbr-source-chat                   2.0
towerchat-ft-7b-wo-context        3.0
towerchat-ft-7b-w-context         2.0
dtype: float64

In [18]:
from sacrebleu import corpus_chrf
from tqdm import tqdm

n_iters = 100
pct_dataset = 0.5


def load_results_chrf(results_df, model, lp, metric):
    lp_subset = results_df[results_df["lp"] == lp]
    references = [str(s) for s in lp_subset["reference"].tolist()]
    hypotheses = [str(s) for s in lp_subset[f"{model}"].tolist()]
    metric_score = corpus_chrf(hypotheses, [references]).score
    return {
        "model": model,
        "lp": lp,
        metric: metric_score,
    }


def load_segment_results_chrf(results_df, model, lp, n_iters, pct_dataset):
    lp_subset = results_df[results_df["lp"] == lp]
    scores = []
    for i in tqdm(range(n_iters), total=n_iters, desc=f"Bootstrapping chrf for {lp}"):
        # make sure we test on different subsets for different models
        random_subset = lp_subset.sample(frac=pct_dataset, random_state=i).reset_index(
            drop=True
        )
        references = [str(s) for s in random_subset["reference"].tolist()]
        hypotheses = [str(s) for s in random_subset[f"{model}"].tolist()]
        scores.append(corpus_chrf(hypotheses, [references]).score)

    return scores


lps = [
    "en-de",
    "de-en",
]
lp_names = [
    "en-de",
    "en-de",
]
from_en_lps = [lp for lp in lps if lp.startswith("en-")]
into_en_lps = [lp for lp in lps if lp.endswith("-en")]
metric = "chrf"
results = pd.concat(
    [
        pd.DataFrame(
            [
                load_results_chrf(
                    pd.read_csv(f"../tacl_review/bcontrast/test.{lp_name}.csv"),
                    model,
                    lp,
                    metric,
                )
                for lp, lp_name in zip(lps, lp_names)
            ]
        )
        for model in models
    ]
)
if metric == "metricx":
    results[metric] = -results[metric]
clusters_by_lp = {}
threshold_p_value = 0.05
for lp, lp_name in zip(lps, lp_names):
    data = {
        m: load_segment_results_chrf(
            pd.read_csv(f"../tacl_review/bcontrast/test.{lp_name}.csv"),
            m,
            lp,
            n_iters,
            pct_dataset,
        )
        for m in models
    }
    seg_scores = pd.DataFrame(data)
    if metric == "metricx":
        seg_scores = -seg_scores

    clusters = rank_systems(seg_scores, threshold_p_value=threshold_p_value)

    clusters_by_lp[lp] = clusters

clusters = pd.DataFrame(clusters_by_lp)

# aggregate and get bordas
pivot_results = results.pivot(columns="lp", index="model", values=metric)
latex_results = pivot_results.round(2)
latex_results = latex_results[lps]

info_by_metric["chrf"] = {
    "latex_results": latex_results,
    "clusters": clusters,
}

Bootstrapping chrf for de-en: 100%|██████████| 100/100 [00:04<00:00, 20.13it/s]


In [19]:
# aggregate and get bordas
pivot_results = results.pivot(columns="lp", index="model", values=metric)
pivot_results = pivot_results.loc[models]
into_en_results = pivot_results.loc[:, into_en_lps].mean(axis=1)
from_en_results = pivot_results.loc[:, from_en_lps].mean(axis=1)

from_en_borda_count = (
    (clusters[from_en_lps].sum(axis=1).sort_values() / len(from_en_lps))
    .round(0)
    .loc[models]
)
into_en_borda_count = (
    (clusters[into_en_lps].sum(axis=1).sort_values() / len(into_en_lps))
    .round(0)
    .loc[models]
)
print(f"From EN borda counts")
display(from_en_borda_count)
print(f"Into EN borda counts")
display(into_en_borda_count)

From EN borda counts


gpt-4o-no-context                  2.0
gpt-4o-context                     1.0
greedy-7b-wo-context              13.0
greedy-7b-w-context               12.0
mbr-instruct                      11.0
mbr-source-instruct               10.0
greedy-7b-ft-wo-context            9.0
greedy-7b-ft-w-context             6.0
greedy-7b-ft-w-context-english     7.0
mbr-chat                           3.0
mbr-source-chat                    4.0
towerchat-ft-7b-wo-context         8.0
towerchat-ft-7b-w-context          5.0
dtype: float64

Into EN borda counts


gpt-4o-no-context                  2.0
gpt-4o-context                     1.0
greedy-7b-wo-context               6.0
greedy-7b-w-context                3.0
mbr-instruct                       8.0
mbr-source-instruct                7.0
greedy-7b-ft-wo-context           10.0
greedy-7b-ft-w-context             7.0
greedy-7b-ft-w-context-english     7.0
mbr-chat                           4.0
mbr-source-chat                    5.0
towerchat-ft-7b-wo-context         9.0
towerchat-ft-7b-w-context          8.0
dtype: float64